# Tổng hợp các mô hình từ đầu đến hiện tại để so sánh các độ đo và đảm bảo tính thực tiễn của các mô hình áp dụng

## **Phương pháp Neural Network áp dụng trong bài toán TicTacToe**

## Môi trường game TTT

In [1]:
import numpy as np
import random
from copy import deepcopy
class action_space:
    def __init__(self, n):
        self.n = n
    
class observation_space:
    def __init__(self, n):
        self.shape = (n,)
class ttt:
    def __init__(self): 
        self.action_space = action_space(9)
        self.observation_space = observation_space(9)
        self.info = ""         
        self.cellcenter = {1:(-200,-200), 2:(0,-200), 3:(200,-200),
                           4:(-200,0),    5:(0,0),    6:(200,0),
                           7:(-200,200),  8:(0,200),  9:(200,200)} 
        self.reset()
        
    def sample(self):
        return random.choice(self.validinputs)   
    def reset(self):  
        self.turn = "X"
        self.rounds = 1
        self.validinputs = list(range(1, 10))
        self.occupied = {"X": [], "O": []}
        self.state = np.array([0]*9)
        self.done = False
        self.reward = 0     
        return self.state        
        
    def step(self, inp):
        inp = int(inp)
        self.occupied[self.turn].append(inp)
        self.state[inp - 1] = 1 if self.turn == "X" else -1
        self.validinputs.remove(inp) 
        
        if self.win_game():
            self.done = True
            self.reward = 1 if self.turn == "X" else -1
            self.validinputs = []
        elif self.rounds == 9:
            self.done = True
            self.reward = 0
            self.validinputs = []
        else:
            self.rounds += 1
            self.turn = "O" if self.turn == "X" else "X"             
        return self.state, self.reward, self.done, self.info
                    
    def win_game(self):
        lst = self.occupied[self.turn]
        lines = [
            [1, 2, 3], [4, 5, 6], [7, 8, 9],
            [1, 4, 7], [2, 5, 8], [3, 6, 9],
            [1, 5, 9], [3, 5, 7]
        ]
        for line in lines:
            if line[0] in lst and line[1] in lst and line[2] in lst:
                return True
        return False
print("✅ Đã khởi tạo môi trường ttt() độc lập, sẵn sàng chạy trên Kaggle!")

✅ Đã khởi tạo môi trường ttt() độc lập, sẵn sàng chạy trên Kaggle!


## Định nghĩa các thuật toán cần thiết: Minimax_ab()

In [2]:
# =============================================================================
# THUẬT TOÁN MINIMAX ALPHA-BETA PRUNING (EXPERT PLAYER CHO TICTACTOE)
# =============================================================================
from copy import deepcopy
from random import choice

def maximized_payoff_ttt(env, reward, done, alpha, beta):
    if done:
        return -1 if reward != 0 else 0
    if alpha is None: alpha = -2
    if beta is None: beta = -2
    
    best_payoff = alpha if env.turn == "X" else beta         
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, info = env_copy.step(m)  
        opponent_payoff = maximized_payoff_ttt(env_copy, reward, done, alpha, beta)
        my_payoff = -opponent_payoff 
        if my_payoff > best_payoff:        
            best_payoff = my_payoff
            if env.turn == "X": alpha = best_payoff
            if env.turn == "O": beta = best_payoff 
        if alpha >= -beta:
            break        
    return best_payoff        

def MiniMax_ab(env):
    wins = []
    ties = []
    losses = []  
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, info = env_copy.step(m) 
        if done and reward != 0:
            return m 
        opponent_payoff = maximized_payoff_ttt(env_copy, reward, done, -2, -2)  
        my_payoff = -opponent_payoff 
        if my_payoff == 1:
            wins.append(m)
        elif my_payoff == 0:
            ties.append(m)
        else:
            losses.append(m)
            
    if len(wins) > 0:
        return choice(wins)
    elif len(ties) > 0:
        return choice(ties)
    return env.sample()

print("✅ Đã khởi tạo hàm MiniMax_ab(env) độc lập!")

✅ Đã khởi tạo hàm MiniMax_ab(env) độc lập!


## Định nghĩa lớp tích chập CNN

In [3]:
import numpy as np

board = np.array([[1,0,0],
                   [1,-1,-1],
                   [1,0,0]]).reshape(-1,3,3,1) 

In [4]:
# Create a vertical filter
vertical_filter = np.array([[0,1,0], 
                   [0,1,0],
                   [0,1,0]]).reshape(3,3,1,1)  

In [5]:
import tensorflow as tf

# Ép kiểu dữ liệu sang tf.float32 để tương thích với tf.nn.conv2d
board_float = tf.cast(board, tf.float32)
filter_float = tf.cast(vertical_filter, tf.float32)

# Áp dụng phép tích chập (Convolution 2D)
result = tf.nn.conv2d(board_float, filter_float, strides=1, padding="SAME")

# In kết quả dạng ma trận 3x3
print(result.numpy().reshape(3, 3))

[[ 2. -1. -1.]
 [ 3. -1. -1.]
 [ 2. -1. -1.]]


2026-09-21 13:27:29.041286: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Tạo dữ liệu mẫu

Tạo dữ liệu mẫu bằng cách sử dụng model cây Minimax làm người chơi chuyên nghiệp, người chơi nghiệp dư với 50% đi giống với cây Minimax, 50% đi ngẫu nhiên

In [6]:
import numpy as np

def expert(env):
    return MiniMax_ab(env)    

def non_expert(env):
    if np.random.rand() < 0.5:
        return MiniMax_ab(env)
    else:
        return env.sample()

In [7]:
from copy import deepcopy

env = ttt()

def one_game(episode):
    history = []
    state = env.reset()  
    # Người chơi non-expert đi trước một nửa số ván (các ván chẵn)
    if episode % 2 == 0:
        action = non_expert(env)
        state, reward, done, _ = env.step(action)
    while True:   
        action = expert(env) 
        if episode % 2 == 0:
            statei = deepcopy(-state)
        else:
            statei = deepcopy(state)            
        actioni = deepcopy(action)
        history.append((statei, actioni))
        state, reward, done, _ = env.step(action)
        if done:
            break
        action = non_expert(env)
        state, reward, done, _ = env.step(action)     
        if done:
            break
    return history

# Test thử nghiệm 1 ván
sample_history = one_game(0)
print(f"Mẫu dữ liệu ghi nhận từ 1 ván: {len(sample_history)} nước đi.")
print(sample_history)

Mẫu dữ liệu ghi nhận từ 1 ván: 4 nước đi.
[(array([ 0,  0, -1,  0,  0,  0,  0,  0,  0]), 5), (array([ 0,  0, -1,  0,  1, -1,  0,  0,  0]), 9), (array([-1,  0, -1,  0,  1, -1,  0,  0,  1]), 2), (array([-1,  1, -1,  0,  1, -1,  0, -1,  1]), 4)]


In [8]:
import os
import pickle
import time

# Đường dẫn thư mục làm việc (Tự nhận diện Kaggle hoặc Local)
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./files"
os.makedirs(WORKING_DIR, exist_ok=True)
GAMES_DATA_PATH = os.path.join(WORKING_DIR, "games_ttt.p")

TOTAL_GAMES = 10000
SAVE_INTERVAL = 1000  # Lưu checkpoint định kỳ mỗi 1.000 ván

# 1. Kiểm tra nếu đã có checkpoint từ trước
results = []
start_episode = 0

if os.path.exists(GAMES_DATA_PATH):
    try:
        with open(GAMES_DATA_PATH, "rb") as fp:
            saved_data = pickle.load(fp)
            if isinstance(saved_data, dict) and "results" in saved_data:
                results = saved_data["results"]
                start_episode = saved_data.get("episode", 0)
            else:
                results = saved_data
                start_episode = TOTAL_GAMES
        print(f"🔄 Tìm thấy dữ liệu checkpoint! Đã có {len(results)} mẫu từ {start_episode}/{TOTAL_GAMES} ván.")
    except Exception as e:
        print(f"⚠️ Lỗi đọc file cũ ({e}), bắt đầu mô phỏng mới...")
        results = []
        start_episode = 0

# 2. Chạy mô phỏng tiếp tục từ start_episode
if start_episode < TOTAL_GAMES:
    print(f"🚀 Bắt đầu mô phỏng từ ván {start_episode + 1} đến {TOTAL_GAMES}...")
    t0 = time.time()
    for episode in range(start_episode, TOTAL_GAMES):
        history = one_game(episode)
        results += history
        
        # Lưu checkpoint định kỳ
        if (episode + 1) % SAVE_INTERVAL == 0 or (episode + 1) == TOTAL_GAMES:
            checkpoint_payload = {
                "results": results,
                "episode": episode + 1
            }
            with open(GAMES_DATA_PATH, "wb") as fp:
                pickle.dump(checkpoint_payload, fp)
            elapsed = time.time() - t0
            print(f"💾 Checkpoint: Đã hoàn thành {episode + 1}/{TOTAL_GAMES} ván | Thu thập: {len(results)} mẫu ({elapsed:.1f}s)")
else:
    print(f"✅ Đã đủ {TOTAL_GAMES} ván ({len(results)} mẫu trạng thái cờ). Không cần mô phỏng lại!")

🚀 Bắt đầu mô phỏng từ ván 1 đến 10000...
💾 Checkpoint: Đã hoàn thành 1000/10000 ván | Thu thập: 3835 mẫu (2412.7s)
💾 Checkpoint: Đã hoàn thành 2000/10000 ván | Thu thập: 7672 mẫu (4166.0s)
💾 Checkpoint: Đã hoàn thành 3000/10000 ván | Thu thập: 11465 mẫu (5905.2s)
💾 Checkpoint: Đã hoàn thành 4000/10000 ván | Thu thập: 15283 mẫu (7675.1s)
💾 Checkpoint: Đã hoàn thành 5000/10000 ván | Thu thập: 19141 mẫu (9449.3s)
💾 Checkpoint: Đã hoàn thành 6000/10000 ván | Thu thập: 22937 mẫu (11208.4s)
💾 Checkpoint: Đã hoàn thành 7000/10000 ván | Thu thập: 26739 mẫu (12966.7s)
💾 Checkpoint: Đã hoàn thành 8000/10000 ván | Thu thập: 30540 mẫu (14769.8s)
💾 Checkpoint: Đã hoàn thành 9000/10000 ván | Thu thập: 34356 mẫu (16543.9s)
💾 Checkpoint: Đã hoàn thành 10000/10000 ván | Thu thập: 38172 mẫu (18321.1s)


```python
# simulate the game 1000 times and record all games
results = []        
for episode in range(100):
    history=one_game(episode)
    results+=history 
```


## Huấn luyện 2 model policy

In [9]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten

fast_model = Sequential()
fast_model.add(Conv2D(filters=128, 
    kernel_size=(3,3),padding="same",activation="relu",
                 input_shape=(3,3,1)))
fast_model.add(Flatten())
fast_model.add(Dense(units=64, activation="relu"))
fast_model.add(Dense(units=64, activation="relu"))
fast_model.add(Dense(9, activation='softmax'))
fast_model.compile(loss='categorical_crossentropy',
                   optimizer='adam', 
                   metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
strong_model = Sequential()
strong_model.add(Conv2D(filters=128, 
    kernel_size=(3,3),padding="same",activation="relu",
                 input_shape=(3,3,1)))
strong_model.add(Flatten())
strong_model.add(Dense(units=64, activation="relu"))
strong_model.add(Dense(units=64, activation="relu"))
strong_model.add(Dense(units=64, activation="relu"))
strong_model.add(Dense(9, activation='softmax'))
strong_model.compile(loss='categorical_crossentropy',
                   optimizer='adam', 
                   metrics=['accuracy'])

```python
import pickle
import numpy as np
with open('files/games_ttt.p','rb') as fp:
    games=pickle.load(fp)

states=[]
actions=[]
for x in games:
    state=x[0]
    action=to_categorical(x[1]-1,9)
    states.append(state)
    actions.append(action)

X=np.array(states).reshape((-1, 3, 3, 1))
y=np.array(actions).reshape((-1, 9))
```

In [11]:
import numpy as np
import pickle
import os
from tensorflow.keras.utils import to_categorical

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./files"
GAMES_DATA_PATH = os.path.join(WORKING_DIR, "games_ttt.p")

# Lấy dữ liệu từ RAM hoặc từ file checkpoint
if 'results' in globals() and len(results) > 0:
    games = results
    print(f"✅ Lấy dữ liệu trực tiếp từ biến 'results' trong RAM: {len(games)} mẫu.")
elif os.path.exists(GAMES_DATA_PATH):
    with open(GAMES_DATA_PATH, 'rb') as fp:
        loaded = pickle.load(fp)
        games = loaded["results"] if isinstance(loaded, dict) and "results" in loaded else loaded
    print(f"✅ Tải dữ liệu từ file {GAMES_DATA_PATH}: {len(games)} mẫu.")
else:
    raise FileNotFoundError("Chưa có dữ liệu mẫu. Hãy chạy Cell 14 để mô phỏng dữ liệu!")

states = []
actions = []
for x in games:
    state = x[0]
    action = to_categorical(x[1] - 1, 9)
    states.append(state)
    actions.append(action)

X = np.array(states).reshape((-1, 3, 3, 1))
y = np.array(actions).reshape((-1, 9))

print(f"📊 Kích thước dữ liệu huấn luyện: X = {X.shape}, y = {y.shape}")

✅ Lấy dữ liệu trực tiếp từ biến 'results' trong RAM: 38172 mẫu.
📊 Kích thước dữ liệu huấn luyện: X = (38172, 3, 3, 1), y = (38172, 9)


```python
# Train the fast policy network for 100 epochs
fast_model.fit(X, y, epochs=100, verbose=1)
fast_model.save('files/fast_ttt.h5')
```

In [12]:
import os
import json
import tensorflow as tf
from tensorflow.keras.models import load_model

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./files"
FAST_MODEL_PATH = os.path.join(WORKING_DIR, "fast_ttt.h5")
FAST_STATE_PATH = os.path.join(WORKING_DIR, "fast_checkpoint.json")
TOTAL_EPOCHS = 100

# Callback tự động lưu checkpoint sau mỗi epoch
class EpochCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, model_path, state_path, save_freq=5):
        super().__init__()
        self.model_path = model_path
        self.state_path = state_path
        self.save_freq = save_freq
        
    def on_epoch_end(self, epoch, logs=None):
        current_ep = epoch + 1
        if current_ep % self.save_freq == 0 or current_ep == TOTAL_EPOCHS:
            self.model.save(self.model_path)
            state = {"completed_epoch": current_ep}
            with open(self.state_path, "w") as f:
                json.dump(state, f)
            print(f"\n💾 [Fast Model] Đã lưu checkpoint tại Epoch {current_ep}/{TOTAL_EPOCHS}")

# Kiểm tra checkpoint đã lưu trước đó
initial_epoch = 0
if os.path.exists(FAST_MODEL_PATH) and os.path.exists(FAST_STATE_PATH):
    try:
        with open(FAST_STATE_PATH, "r") as f:
            state = json.load(f)
            initial_epoch = state.get("completed_epoch", 0)
        if initial_epoch > 0:
            fast_model = load_model(FAST_MODEL_PATH)
            print(f"🔄 Đã tải Fast Model checkpoint từ Epoch {initial_epoch}!")
    except Exception as e:
        print(f"Không thể tải checkpoint ({e}), khởi động huấn luyện mới.")
        initial_epoch = 0

if initial_epoch >= TOTAL_EPOCHS:
    print(f"✅ Fast Model đã hoàn tất toàn bộ {TOTAL_EPOCHS} epochs từ trước!")
else:
    print(f"🚀 Bắt đầu huấn luyện Fast Model từ Epoch {initial_epoch + 1} đến {TOTAL_EPOCHS}...")
    checkpoint_cb = EpochCheckpoint(FAST_MODEL_PATH, FAST_STATE_PATH, save_freq=5)
    fast_model.fit(
        X, y, 
        epochs=TOTAL_EPOCHS, 
        initial_epoch=initial_epoch, 
        callbacks=[checkpoint_cb],
        verbose=1
    )
    fast_model.save(FAST_MODEL_PATH)
    print(f"✅ Đã lưu Fast Model hoàn chỉnh tại: {FAST_MODEL_PATH}")

🚀 Bắt đầu huấn luyện Fast Model từ Epoch 1 đến 100...
Epoch 1/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5326 - loss: 1.2203
Epoch 2/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6086 - loss: 0.9339
Epoch 3/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6173 - loss: 0.8908
Epoch 4/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6194 - loss: 0.8719
Epoch 5/100
1181/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6254 - loss: 0.8544


💾 [Fast Model] Đã lưu checkpoint tại Epoch 5/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6228 - loss: 0.8568
Epoch 6/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6244 - loss: 0.8506
Epoch 7/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6258 - loss: 0.8412
Epoch 8/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6268 - loss: 0.8337
Epoch 9/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6252 - loss: 0.8308
Epoch 10/100
1187/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6330 - loss: 0.8184


💾 [Fast Model] Đã lưu checkpoint tại Epoch 10/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6294 - loss: 0.8238
Epoch 11/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6304 - loss: 0.8222
Epoch 12/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6294 - loss: 0.8196
Epoch 13/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6302 - loss: 0.8145
Epoch 14/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6309 - loss: 0.8124
Epoch 15/100
1187/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6364 - loss: 0.7985


💾 [Fast Model] Đã lưu checkpoint tại Epoch 15/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6306 - loss: 0.8099
Epoch 16/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6317 - loss: 0.8072
Epoch 17/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6303 - loss: 0.8050
Epoch 18/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6300 - loss: 0.8034
Epoch 19/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6316 - loss: 0.8015
Epoch 20/100
1187/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6351 - loss: 0.7949


💾 [Fast Model] Đã lưu checkpoint tại Epoch 20/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6334 - loss: 0.7995
Epoch 21/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6319 - loss: 0.7976
Epoch 22/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6310 - loss: 0.7969
Epoch 23/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6321 - loss: 0.7956
Epoch 24/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6326 - loss: 0.7942
Epoch 25/100
1180/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6350 - loss: 0.7863

...


💾 [Fast Model] Đã lưu checkpoint tại Epoch 85/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6413 - loss: 0.7635
Epoch 86/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6427 - loss: 0.7656
Epoch 87/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6415 - loss: 0.7637
Epoch 88/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6422 - loss: 0.7615
Epoch 89/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6414 - loss: 0.7636
Epoch 90/100
1184/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6442 - loss: 0.7648


💾 [Fast Model] Đã lưu checkpoint tại Epoch 90/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6408 - loss: 0.7631
Epoch 91/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6423 - loss: 0.7632
Epoch 92/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6387 - loss: 0.7617
Epoch 93/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6417 - loss: 0.7636
Epoch 94/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6431 - loss: 0.7623
Epoch 95/100
1179/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6451 - loss: 0.7627


💾 [Fast Model] Đã lưu checkpoint tại Epoch 95/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6405 - loss: 0.7663
Epoch 96/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6431 - loss: 0.7622
Epoch 97/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6430 - loss: 0.7612
Epoch 98/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6424 - loss: 0.7645
Epoch 99/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6417 - loss: 0.7643
Epoch 100/100
1190/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6427 - loss: 0.7637


💾 [Fast Model] Đã lưu checkpoint tại Epoch 100/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6415 - loss: 0.7611


✅ Đã lưu Fast Model hoàn chỉnh tại: /kaggle/working/fast_ttt.h5


```python
strong_model.fit(X, y, epochs=100, verbose=1)
strong_model.save('files/strong_ttt.h5')
```

In [13]:
import os
import json
import tensorflow as tf
from tensorflow.keras.models import load_model

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./files"
STRONG_MODEL_PATH = os.path.join(WORKING_DIR, "strong_ttt.h5")
STRONG_STATE_PATH = os.path.join(WORKING_DIR, "strong_checkpoint.json")
TOTAL_EPOCHS = 100

initial_epoch = 0
if os.path.exists(STRONG_MODEL_PATH) and os.path.exists(STRONG_STATE_PATH):
    try:
        with open(STRONG_STATE_PATH, "r") as f:
            state = json.load(f)
            initial_epoch = state.get("completed_epoch", 0)
        if initial_epoch > 0:
            strong_model = load_model(STRONG_MODEL_PATH)
            print(f"🔄 Đã tải Strong Model checkpoint từ Epoch {initial_epoch}!")
    except Exception as e:
        print(f"Không thể tải checkpoint ({e}), khởi động huấn luyện mới.")
        initial_epoch = 0

if initial_epoch >= TOTAL_EPOCHS:
    print(f"✅ Strong Model đã hoàn tất toàn bộ {TOTAL_EPOCHS} epochs từ trước!")
else:
    print(f"🚀 Bắt đầu huấn luyện Strong Model từ Epoch {initial_epoch + 1} đến {TOTAL_EPOCHS}...")
    checkpoint_cb = EpochCheckpoint(STRONG_MODEL_PATH, STRONG_STATE_PATH, save_freq=5)
    strong_model.fit(
        X, y, 
        epochs=TOTAL_EPOCHS, 
        initial_epoch=initial_epoch, 
        callbacks=[checkpoint_cb],
        verbose=1
    )
    strong_model.save(STRONG_MODEL_PATH)
    print(f"✅ Đã lưu Strong Model hoàn chỉnh tại: {STRONG_MODEL_PATH}")

🚀 Bắt đầu huấn luyện Strong Model từ Epoch 1 đến 100...
Epoch 1/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5138 - loss: 1.2708
Epoch 2/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6077 - loss: 0.9372
Epoch 3/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6157 - loss: 0.8918
Epoch 4/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6210 - loss: 0.8685
Epoch 5/100
1189/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6262 - loss: 0.8520


💾 [Fast Model] Đã lưu checkpoint tại Epoch 5/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6249 - loss: 0.8528
Epoch 6/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6253 - loss: 0.8442
Epoch 7/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6273 - loss: 0.8354
Epoch 8/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6255 - loss: 0.8307
Epoch 9/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6282 - loss: 0.8245
Epoch 10/100
1189/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6267 - loss: 0.8237


💾 [Fast Model] Đã lưu checkpoint tại Epoch 10/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6277 - loss: 0.8215
Epoch 11/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6276 - loss: 0.8160
Epoch 12/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6320 - loss: 0.8117
Epoch 13/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6293 - loss: 0.8106
Epoch 14/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6323 - loss: 0.8070
Epoch 15/100
1192/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6385 - loss: 0.7973


💾 [Fast Model] Đã lưu checkpoint tại Epoch 15/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6330 - loss: 0.8031
Epoch 16/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6317 - loss: 0.8023
Epoch 17/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6314 - loss: 0.7996
Epoch 18/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6325 - loss: 0.7970
Epoch 19/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6323 - loss: 0.7966
Epoch 20/100
1189/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6354 - loss: 0.7873


💾 [Fast Model] Đã lưu checkpoint tại Epoch 20/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6338 - loss: 0.7918
Epoch 21/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6310 - loss: 0.7908
Epoch 22/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6314 - loss: 0.7899
Epoch 23/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6347 - loss: 0.7875
Epoch 24/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6323 - loss: 0.7870
Epoch 25/100
1183/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6354 - loss: 0.7873

...


💾 [Fast Model] Đã lưu checkpoint tại Epoch 85/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6422 - loss: 0.7671
Epoch 86/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6405 - loss: 0.7613
Epoch 87/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6412 - loss: 0.7627
Epoch 88/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6410 - loss: 0.7677
Epoch 89/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6409 - loss: 0.7639
Epoch 90/100
1182/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6443 - loss: 0.7610


💾 [Fast Model] Đã lưu checkpoint tại Epoch 90/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6423 - loss: 0.7631
Epoch 91/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6434 - loss: 0.7612
Epoch 92/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6383 - loss: 0.7658
Epoch 93/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6415 - loss: 0.7686
Epoch 94/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6417 - loss: 0.7620
Epoch 95/100
1192/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6463 - loss: 0.7542


💾 [Fast Model] Đã lưu checkpoint tại Epoch 95/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6400 - loss: 0.7658
Epoch 96/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6410 - loss: 0.7669
Epoch 97/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6435 - loss: 0.7603
Epoch 98/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6431 - loss: 0.7608
Epoch 99/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6423 - loss: 0.7674
Epoch 100/100
1185/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6440 - loss: 0.7668


💾 [Fast Model] Đã lưu checkpoint tại Epoch 100/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6431 - loss: 0.7643


✅ Đã lưu Strong Model hoàn chỉnh tại: /kaggle/working/strong_ttt.h5


## Chạy thử nghiệm

**Môi trường util10**

In [14]:
# =============================================================================
# MÔI TRƯỜNG CỐT LÕI MCTS & CÔNG THỨC UCT TÍCH HỢP POLICY NETWORK
# =============================================================================
import random
from copy import deepcopy
from math import sqrt, log
import numpy as np

# 1. Các bước cơ bản của MCTS truyền thống (Chương 8)
def expand(env, move):
    env_copy = deepcopy(env)
    state, reward, done, info = env_copy.step(move)
    return env_copy, done, reward

def simulate(env_copy, done, reward):
    if done:
        return reward
    while True:
        move = env_copy.sample()
        state, reward, done, info = env_copy.step(move)
        if done:
            return reward

def backpropagate(env, move, reward, counts, wins, losses):
    counts[move] = counts.get(move, 0) + 1
    if reward == 1 and env.turn == "X":
        wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "O":
        wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "X":
        losses[move] = losses.get(move, 0) + 1
    elif reward == 1 and env.turn == "O":
        losses[move] = losses.get(move, 0) + 1
    return counts, wins, losses

def select(env, counts, wins, losses, temperature=1.414):
    for k in env.validinputs:
        if counts[k] == 0:
            return k
    N = sum(counts.values())
    scores = {}
    for k in env.validinputs:
        vi = (wins.get(k, 0) - losses.get(k, 0)) / counts[k]
        exploration = temperature * sqrt(log(N) / counts[k])
        scores[k] = vi + exploration
    return max(scores, key=scores.get)

def next_best_move(counts, wins, losses):
    scores = {}
    for k in counts.keys():
        if counts[k] == 0:
            scores[k] = -float('inf')
        else:
            scores[k] = (wins.get(k, 0) - losses.get(k, 0)) / counts[k]
    return max(scores, key=scores.get)

def mcts(env, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    for _ in range(num_rollouts):
        move = select(env, counts, wins, losses, temperature)
        env_copy, done, reward = expand(env, move)
        reward = simulate(env_copy, done, reward)
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
    return next_best_move(counts, wins, losses)

# 2. Các hàm mở rộng kết hợp Policy Network (Chương 10)
gamma = 10  # Hệ số cân bằng trọng số mạng Policy

def mix_select(env, ps, counts, wins, losses, temperature=1.414):
    for k in env.validinputs:
        if counts[k] == 0:
            return k
    N = sum(counts.values())
    scores = {}
    for k in env.validinputs:
        weighted_pi = gamma * ps[k] / (1 + counts[k])
        vi = (wins.get(k, 0) - losses.get(k, 0)) / counts[k]
        exploration = temperature * sqrt(log(N) / counts[k])
        scores[k] = vi + exploration + weighted_pi
    return max(scores, key=scores.get)

def next_move_policy(ps, counts, wins, losses):
    scores = {}
    for k, v in counts.items():
        weighted_pi = gamma * ps[k] / (1 + counts[k])
        vi = (wins.get(k, 0) - losses.get(k, 0)) / v if v > 0 else 0
        scores[k] = vi + weighted_pi
    return max(scores, key=scores.get)

print("✅ Đã cài đặt xong toàn bộ môi trường cốt lõi MCTS & Policy UCT!")

✅ Đã cài đặt xong toàn bộ môi trường cốt lõi MCTS & Policy UCT!


**Mixed MCTS v1**: Chỉ implement code áp dụng strong policy network cho bước selection của MCTS

In [15]:
# =============================================================================
# MIXED MCTS V1: STRONG POLICY CHO SELECTION + ROLLOUT NGẪU NHIÊN
# =============================================================================
def mix_mcts_v1(env, model, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
        
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    
    # 1. Dự đoán phân phối xác suất từ Strong Policy Network
    state = env.state.reshape(-1, 3, 3, 1)
    if env.turn == "X":
        action_probs = model(state, training=False).numpy()
    else:
        action_probs = model(-state, training=False).numpy()
    
    ps = {a: float(np.squeeze(action_probs)[a - 1]) for a in env.validinputs}
    
    # 2. Chạy các lượt mô phỏng Rollouts
    for _ in range(num_rollouts):
        # Bước 1: Selection (kết hợp Strong Policy)
        move = mix_select(env, ps, counts, wins, losses, temperature)
        # Bước 2: Expansion
        env_copy, done, reward = expand(env, move)
        # Bước 3: Simulation (Ngẫu nhiên thuần túy)
        reward = simulate(env_copy, done, reward)
        # Bước 4: Backpropagation
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
        
    return next_move_policy(ps, counts, wins, losses)

# Thử nghiệm nhanh 1 nước đi
test_env = ttt()
move_v1 = mix_mcts_v1(test_env, strong_model, num_rollouts=100)
print(f"🎯 [Mixed MCTS v1] Nước đi đề xuất cho bàn cờ trống: Ô số {move_v1}")

🎯 [Mixed MCTS v1] Nước đi đề xuất cho bàn cờ trống: Ô số 9


**Mixed MCTS v2**: Chỉ implement code áp dụng fast policy network cho bước rollout của MCTS

In [16]:
# =============================================================================
# MIXED MCTS V2: UCT CHO SELECTION + FAST POLICY CHO ROLLOUT
# =============================================================================
def simulate_fast_policy(env_copy, done, reward, model):
    if done:
        return reward
    while True:
        state = env_copy.state.reshape(-1, 3, 3, 1)
        if env_copy.turn == "X":
            probs = model(state, training=False).numpy().flatten()
        else:
            probs = model(-state, training=False).numpy().flatten()
            
        valid_moves = env_copy.validinputs
        valid_probs = np.array([probs[m - 1] for m in valid_moves])
        prob_sum = valid_probs.sum()
        
        if prob_sum > 0:
            valid_probs = valid_probs / prob_sum
            move = np.random.choice(valid_moves, p=valid_probs)
        else:
            move = random.choice(valid_moves)
            
        state, reward, done, info = env_copy.step(move)
        if done:
            return reward

def mix_mcts_v2(env, model, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
        
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    
    for _ in range(num_rollouts):
        # Bước 1: Selection (UCT chuẩn)
        move = select(env, counts, wins, losses, temperature)
        # Bước 2: Expansion
        env_copy, done, reward = expand(env, move)
        # Bước 3: Simulation (Dùng Fast Policy Network)
        reward = simulate_fast_policy(env_copy, done, reward, model)
        # Bước 4: Backpropagation
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
        
    return next_best_move(counts, wins, losses)

# Thử nghiệm nhanh 1 nước đi
test_env = ttt()
move_v2 = mix_mcts_v2(test_env, fast_model, num_rollouts=100)
print(f"🎯 [Mixed MCTS v2] Nước đi đề xuất cho bàn cờ trống: Ô số {move_v2}")

🎯 [Mixed MCTS v2] Nước đi đề xuất cho bàn cờ trống: Ô số 1


**Mixed MCTS v3**: Chỉ implement code, áp dụng cả 2 strong policy network cho bước selection của MCTS và fast policy network cho bước rollout của MCTS

In [17]:
# =============================================================================
# MIXED MCTS V3: STRONG POLICY CHO SELECTION + FAST POLICY CHO ROLLOUT
# =============================================================================
def mix_mcts_v3(env, strong_model, fast_model, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
        
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    
    # 1. Tính toán xác suất Tiên nghiệm từ Strong Policy Network
    state = env.state.reshape(-1, 3, 3, 1)
    if env.turn == "X":
        action_probs = strong_model(state, training=False).numpy()
    else:
        action_probs = strong_model(-state, training=False).numpy()
    
    ps = {a: float(np.squeeze(action_probs)[a - 1]) for a in env.validinputs}
    
    # 2. Rollout mô phỏng
    for _ in range(num_rollouts):
        # Bước 1: Selection (kết hợp Strong Policy)
        move = mix_select(env, ps, counts, wins, losses, temperature)
        # Bước 2: Expansion
        env_copy, done, reward = expand(env, move)
        # Bước 3: Simulation (Định hướng bởi Fast Policy)
        reward = simulate_fast_policy(env_copy, done, reward, fast_model)
        # Bước 4: Backpropagation
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
        
    return next_move_policy(ps, counts, wins, losses)

# Thử nghiệm nhanh 1 nước đi
test_env = ttt()
move_v3 = mix_mcts_v3(test_env, strong_model, fast_model, num_rollouts=100)
print(f"🎯 [Mixed MCTS v3 (AlphaGo)] Nước đi đề xuất cho bàn cờ trống: Ô số {move_v3}")

# Đánh giá thi đấu nhanh giữa Mixed MCTS v3 và MCTS thuần
print("\n⚔️ Chạy thử nghiệm đối đầu: Mixed MCTS v3 vs MCTS thuần (20 ván)...")
v3_wins, ties, mcts_wins = 0, 0, 0
for i in range(20):
    g_env = ttt()
    while True:
        # Lượt 1: v3 (X)
        act = mix_mcts_v3(g_env, strong_model, fast_model, num_rollouts=80)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == 1: v3_wins += 1
            else: ties += 1
            break
        # Lượt 2: MCTS truyền thống (O)
        act = mcts(g_env, num_rollouts=80)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == -1: mcts_wins += 1
            else: ties += 1
            break

print(f"🏆 Kết quả sau 20 ván: Mixed MCTS v3 Thắng {v3_wins} | Hòa {ties} | MCTS Thắng {mcts_wins}")

🎯 [Mixed MCTS v3 (AlphaGo)] Nước đi đề xuất cho bàn cờ trống: Ô số 2

⚔️ Chạy thử nghiệm đối đầu: Mixed MCTS v3 vs MCTS thuần (20 ván)...
🏆 Kết quả sau 20 ván: Mixed MCTS v3 Thắng 17 | Hòa 3 | MCTS Thắng 0
